In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import Dataset
import torch

In [2]:
class SFTDataset:
    def __init__(self, tokenizer, instructions, responses, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.example = []
        
        for inst, resp in zip(instructions, responses):
            text = f"### Instruction:\n{inst}\n\n### Response:\n{resp}"
            self.example.append(text)
    
    def __len__(self):
        return len(self.example)
    
    def __getitem__(self, idx):
        text = self.example[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].squeeze()
        labels = input_ids.clone()
        
        instruction_end = text.find("### Response:")
        instruction_tokens = self.tokenizer(
            text[:instruction_end],
            max_length=self.max_length,
            truncation=True,
            return_tensors='pt'
        )['input_ids'].squeeze()
        
        labels[:len(instruction_tokens)] = -100
        
        return {
            'input_ids': input_ids,
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': labels
        }

In [7]:
def train_sft(model_name, train_data, output_dir='./sft_model'):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    # Prepare dataset
    instructions = [item['instruction'] for item in train_data]
    responses = [item['response'] for item in train_data]
    dataset = SFTDataset(tokenizer, instructions, responses)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        warmup_steps=100,
        logging_steps=1,
        save_steps=500,
        fp16=True,
        remove_unused_columns=False
    )
    
    def data_collator(features):
        batch = {
            'input_ids' : torch.stack([f['input_ids'] for f in features]),
            'attention_mask' : torch.stack([f['attention_mask'] for f in features]),
            'labels' : torch.stack([f['labels'] for f in features])
        }
        return batch

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=data_collator
    )
    
    trainer.train()
    trainer.save_model()
    
    return model, tokenizer

In [5]:
train_data = [
    {"instruction": "What is Python?", "response": "Python is a high-level programming language known for its simplicity and readability."},
    {"instruction": "What is a variable in programming?", "response": "A variable is a named container used to store data values in a program."},
    {"instruction": "What is a function?", "response": "A function is a reusable block of code that performs a specific task."},
    {"instruction": "What is a list in Python?", "response": "A list in Python is an ordered, mutable collection that can store multiple items."},
    {"instruction": "What is a dictionary in Python?", "response": "A dictionary is a key-value data structure used to store data in pairs."},
    {"instruction": "What is a loop?", "response": "A loop is a control structure that repeats a block of code multiple times."},
    {"instruction": "What is an if statement?", "response": "An if statement is used to make decisions in code based on a condition."},
    {"instruction": "What is object-oriented programming?", "response": "Object-oriented programming is a programming style based on objects, classes, and methods."},
    {"instruction": "What is machine learning?", "response": "Machine learning is a field of AI where systems learn patterns from data to make predictions or decisions."},
    {"instruction": "What is a neural network?", "response": "A neural network is a computational model inspired by the human brain, often used in deep learning."}
]

In [8]:
model, tokenizer = train_sft("gpt2", train_data)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
1,10.100472
2,10.115026
3,10.109497


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]